# VMGP: Variational Auto-Encoder Multi-task Genomic Prediction
## Training & Risultati

Questo notebook utilizza il modulo `vae/` per training e analisi.

### Struttura del Modulo:
- `vae/network.py` - Architettura VMGP
- `vae/lightning_module.py` - Logic training (PyTorch Lightning)
- `vae/data_module.py` - Caricamento e preprocessing dati PLINK
- `vae/utils.py` - Funzioni di utilità (balancing, capping, metriche)
- `vae/experiment.py` - Runner esperimenti con K-Fold CV
- `vae/analysis.py` - Analisi post-training

### Esperimento:
Griglia completa di **20 combinazioni**:
- **Min Samples per Razza:** 30, 40, 50, 60, 70 (solo sample reali, NO SMOTE)
- **Dimensione Latent Space:** 32, 64, 96, 128
- **Task:** Solo classificazione breed

---

## 1. Setup e Imports

In [2]:
import os
import sys
import torch
import pandas as pd
import pytorch_lightning as pl

# Aggiungi la directory parent al path per importare il modulo vae
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Imposta seed per riproducibilità
pl.seed_everything(42)
# Import modulo VAE
from vae import (
    VMGP_Network,
    VMGP_LightningSystem,
    GenomicDataModule,
    run_experiment,
    CLASSIFIER_CONFIGS,
    cap_dataset_multilabel,
    analyze_model_performance
)

print("✅ Moduli caricati correttamente!")
print(f"🔧 CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

Seed set to 42


✅ Moduli caricati correttamente!
🔧 CUDA disponibile: True
   GPU: NVIDIA GeForce RTX 5070 Ti


## 2. Configurazione

In [3]:
# ⚠️ MODIFICA QUESTI PATH CON IL TUO DATASET ⚠️
BED_PATH = "../data/ADAPTmap_genotypeTOP_20160222_full.bed"
METADATA_PATH = "classificazione-capre.csv"

# Hyperparameters
config = {
    # DataModule
    'bed_path': BED_PATH,
    'metadata_path': METADATA_PATH,
    'batch_size': 64,
    'maf_thresh': 0.01,
    'geno_thresh': 0.1,
    'min_samples_per_class': 30,  # Verrà variato negli esperimenti
    'ld_pruning': True,
    'ld_threshold': 0.2,
    'ld_window': 50,
    'val_split': 0.15,
    'test_split': 0.15,
    
    # Cap samples (solo downsampling, NO dati sintetici)
    'target_samples': 30,  # Verrà variato negli esperimenti
    
    # Model (VMGP) - Loss weights
    'alpha': 0.5,
    'lr': 0.0001,
    'latent_dim': 32,  # Default, verrà variato negli esperimenti
    
    # Solo classificatore breed
    'weight_breed': 1.0,
    'weight_continent': 0.0,
    'weight_caseina': 0.0,
    
    # Training
    'max_epochs': 100,
    'k_folds': 3,  # 3-Fold Cross Validation
    'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
    'devices': 1,
}

# Griglia esperimenti
MIN_SAMPLES_RANGE = [30, 40, 50, 60, 70]   # 5 valori
LATENT_DIM_RANGE = [32, 64, 96, 128]        # 4 valori
# Totale: 5 × 4 = 20 combinazioni

print("📋 CONFIGURAZIONE BASE:")
for k, v in config.items():
    print(f"   {k}: {v}")

print(f"\n📊 GRIGLIA ESPERIMENTI:")
print(f"   Min Samples: {MIN_SAMPLES_RANGE}")
print(f"   Latent Dims: {LATENT_DIM_RANGE}")
print(f"   K-Fold CV: {config['k_folds']} fold")
print(f"   Totale combinazioni: {len(MIN_SAMPLES_RANGE) * len(LATENT_DIM_RANGE)}")
print(f"   Task: Solo classificazione breed (NO continente, NO caseina)")
print(f"   Dati: Solo sample reali (NO SMOTE)")

# Conteggio razze disponibili per ogni soglia
print(f"\n📈 RAZZE DISPONIBILI PER SOGLIA (dal dataset ADAPTmap):")
print(f"   ≥30 samples: 34 razze")
print(f"   ≥40 samples: 26 razze")
print(f"   ≥50 samples: 16 razze")
print(f"   ≥60 samples: 12 razze")
print(f"   ≥70 samples: 10 razze")

📋 CONFIGURAZIONE BASE:
   bed_path: ../data/ADAPTmap_genotypeTOP_20160222_full.bed
   metadata_path: classificazione-capre.csv
   batch_size: 64
   maf_thresh: 0.01
   geno_thresh: 0.1
   min_samples_per_class: 30
   ld_pruning: True
   ld_threshold: 0.2
   ld_window: 50
   val_split: 0.15
   test_split: 0.15
   target_samples: 30
   alpha: 0.5
   lr: 0.0001
   latent_dim: 32
   weight_breed: 1.0
   weight_continent: 0.0
   weight_caseina: 0.0
   max_epochs: 100
   k_folds: 3
   accelerator: gpu
   devices: 1

📊 GRIGLIA ESPERIMENTI:
   Min Samples: [30, 40, 50, 60, 70]
   Latent Dims: [32, 64, 96, 128]
   K-Fold CV: 3 fold
   Totale combinazioni: 20
   Task: Solo classificazione breed (NO continente, NO caseina)
   Dati: Solo sample reali (NO SMOTE)

📈 RAZZE DISPONIBILI PER SOGLIA (dal dataset ADAPTmap):
   ≥30 samples: 34 razze
   ≥40 samples: 26 razze
   ≥50 samples: 16 razze
   ≥60 samples: 12 razze
   ≥70 samples: 10 razze


## 3. Training Esperimenti

Seleziona quali esperimenti eseguire decommentando le celle corrispondenti.

In [ ]:
# Container per i risultati
all_results = []

# === SALVATAGGIO INCREMENTALE ===
import json
from pathlib import Path

RESULTS_CACHE_PATH = Path("results_cache.json")

def save_results_incrementally(results_list):
    """Salva i risultati in un file JSON per recupero in caso di crash."""
    # Converti eventuali valori numpy in Python nativi
    clean_results = []
    for res in results_list:
        clean_res = {}
        for k, v in res.items():
            if hasattr(v, 'item'):  # numpy scalar
                clean_res[k] = v.item()
            elif isinstance(v, (list, tuple)):
                clean_res[k] = [x.item() if hasattr(x, 'item') else x for x in v]
            else:
                clean_res[k] = v
        clean_results.append(clean_res)
    
    with open(RESULTS_CACHE_PATH, 'w') as f:
        json.dump(clean_results, f, indent=2)
    print(f"💾 Risultati salvati in {RESULTS_CACHE_PATH}")

def load_cached_results():
    """Carica risultati salvati da file JSON."""
    if RESULTS_CACHE_PATH.exists():
        with open(RESULTS_CACHE_PATH, 'r') as f:
            return json.load(f)
    return []

print("✅ Funzioni di salvataggio incrementale definite")

### 3.1 Esperimento: Griglia Min Samples × Latent Dim

**Obiettivo:** Trovare la combinazione ottimale tra numero minimo di sample per razza e dimensione del latent space.

**Griglia:**
- **Min Samples:** 30, 40, 50, 60, 70 (solo razze con almeno N campioni, cappate a N)
- **Latent Dims:** 32, 64, 96, 128
- **Task:** Solo classificazione breed (senza continente/caseina/attitudine)
- **Dati:** Solo sample reali (NO SMOTE, NO dati sintetici)
- **Totale:** 20 esperimenti

In [ ]:
# === ESPERIMENTO: GRIGLIA MIN SAMPLES × LATENT DIM ===
# 20 combinazioni totali (5 min_samples × 4 latent_dims)
# Solo breed classification, solo dati reali (cap, NO SMOTE)

# --- PRE-CARICAMENTO DATAMODULE (1 volta per min_samples) ---
# Evita di rifare QC + LD Pruning per ogni latent_dim (fix issue #5)
print("📦 Pre-caricamento DataModule per ogni soglia min_samples...")
print("   (QC + LD Pruning eseguiti UNA sola volta per soglia)\n")

cached_data_modules = {}
for min_samples in MIN_SAMPLES_RANGE:
    print(f"   Caricamento min_samples={min_samples}...")
    dm = GenomicDataModule(
        bed_path=config['bed_path'],
        batch_size=config['batch_size'],
        maf_thresh=config['maf_thresh'],
        geno_thresh=config['geno_thresh'],
        min_samples_per_class=min_samples,
        ld_pruning=config['ld_pruning'],
        ld_threshold=config['ld_threshold'],
        ld_window=config['ld_window'],
        val_split=config['val_split'],
        test_split=config['test_split'],
        metadata_path=config['metadata_path'],
        use_breed=True,
        use_continent=False,
        use_caseina=False,
        use_attitudine=False
    )
    dm.setup()
    cached_data_modules[min_samples] = dm
    print(f"   ✅ min_samples={min_samples}: {dm.num_classes_breed} razze, {dm.num_snps} SNPs\n")

print(f"📦 Pre-caricamento completato! ({len(cached_data_modules)} DataModule cachati)")

# --- ESECUZIONE GRIGLIA ---
grid_results = []
total_experiments = len(MIN_SAMPLES_RANGE) * len(LATENT_DIM_RANGE)
exp_counter = 0

for min_samples in MIN_SAMPLES_RANGE:
    dm_cached = cached_data_modules[min_samples]
    
    for latent_dim in LATENT_DIM_RANGE:
        exp_counter += 1
        
        # Crea config specifica per questo esperimento
        exp_config = config.copy()
        exp_config['latent_dim'] = latent_dim
        exp_config['min_samples_per_class'] = min_samples
        exp_config['target_samples'] = min_samples  # Cap a min_samples (solo reali)
        
        exp_name = f"Samples_{min_samples}_LatDim_{latent_dim}"
        
        print(f"\n{'='*60}")
        print(f"🔬 [{exp_counter}/{total_experiments}] Esperimento: {exp_name}")
        print(f"   Min Samples: {min_samples} | Latent Dim: {latent_dim} | K-Fold: {config['k_folds']}")
        print(f"{'='*60}")
        
        try:
            res = run_experiment(
                name=exp_name, 
                config=exp_config,
                balanced=False,        # NO SMOTE
                cap_samples=True,      # Solo downsampling a min_samples
                coarse_mapping=None, 
                max_epochs=config['max_epochs'],
                k_folds=config['k_folds'],  # 3-Fold Cross Validation
                classifier_config='breed_only',
                data_module=dm_cached  # Riusa DataModule pre-caricato (skip QC+LD)
            )
            
            res['Min_Samples'] = min_samples
            res['Latent_Dim'] = latent_dim
            grid_results.append(res)
            all_results.append(res)
            
            # 💾 Salvataggio incrementale dopo ogni esperimento
            save_results_incrementally(all_results)
            
        except Exception as e:
            print(f"⚠️ Errore nell'esperimento {exp_name}: {e}")
            import traceback
            traceback.print_exc()
            continue

print(f"\n✅ Griglia esperimenti completata! ({len(grid_results)}/{total_experiments} riusciti)")

### 3.2 Analisi Risultati Griglia

Heatmap e confronto delle performance per tutte le 20 combinazioni.

In [ ]:
# === ANALISI GRIGLIA MIN SAMPLES × LATENT DIM ===
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

df_grid = pd.DataFrame(grid_results)

# Metriche da visualizzare come heatmap (include Kappa per confronto equo)
metrics_to_plot = {
    'Acc_Breed (mean)': 'Breed Accuracy (raw)',
    'Kappa_Breed (mean)': "Cohen's Kappa (chance-adjusted)",
    'Silhouette (mean)': 'Silhouette Score (Clustering)',
    'Generalization (GE) (mean)': 'Generalization (GE)'
}

# Filtra solo le metriche disponibili
metrics_to_plot = {k: v for k, v in metrics_to_plot.items() if k in df_grid.columns}

n_metrics = len(metrics_to_plot)
n_rows = (n_metrics + 1) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(16, 6 * n_rows))
axes = axes.flatten()

for idx, (metric_col, metric_name) in enumerate(metrics_to_plot.items()):
    ax = axes[idx]
    
    # Crea pivot table per heatmap
    pivot = df_grid.pivot_table(
        values=metric_col, 
        index='Min_Samples', 
        columns='Latent_Dim', 
        aggfunc='mean'
    )
    
    # Colormap: per MSE usiamo reversed (basso è meglio), per le altre no
    cmap = 'RdYlGn' if 'MSE' not in metric_name else 'RdYlGn_r'
    
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap=cmap, ax=ax, 
                linewidths=0.5, linecolor='gray')
    ax.set_title(metric_name, fontweight='bold', fontsize=12)
    ax.set_xlabel('Latent Dimension')
    ax.set_ylabel('Min Samples per Breed')

# Nascondi assi extra
for idx in range(n_metrics, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('📊 Griglia Esperimenti: Min Samples × Latent Dim\n(Solo dati reali, solo breed classification)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grid_experiment_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabella riassuntiva
print("\n📋 TABELLA COMPLETA DEI RISULTATI:")
display_cols = ['Experiment', 'Min_Samples', 'Latent_Dim', 'Num Classes Breed', 'Num SNPs', 'Effective Batch Size']
for m in metrics_to_plot.keys():
    display_cols.append(m)
available_cols = [c for c in display_cols if c in df_grid.columns]
print(df_grid[available_cols].to_string())

# Nota su confronto equo
print("\n⚠️ NOTA SUL CONFRONTO EQUO:")
print("   L'Accuracy raw NON è direttamente comparabile tra soglie diverse")
print("   (classificare 10 razze è più facile che classificarne 34).")
print("   → Usa Cohen's Kappa per un confronto equo (chance-adjusted).")

# Identificazione migliore combinazione
if 'Kappa_Breed (mean)' in df_grid.columns:
    best_idx = df_grid['Kappa_Breed (mean)'].idxmax()
    best = df_grid.loc[best_idx]
    print(f"\n🎯 MIGLIOR COMBINAZIONE (per Cohen's Kappa - confronto equo):")
    print(f"   Min Samples = {best['Min_Samples']}, Latent Dim = {best['Latent_Dim']}")
    print(f"   Kappa: {best['Kappa_Breed (mean)']:.4f}")
    print(f"   Accuracy: {best.get('Acc_Breed (mean)', 'N/A'):.4f}")
    print(f"   Num Razze: {best.get('Num Classes Breed', 'N/A')}")

if 'Acc_Breed (mean)' in df_grid.columns:
    best_acc_idx = df_grid['Acc_Breed (mean)'].idxmax()
    best_acc = df_grid.loc[best_acc_idx]
    print(f"\n🎯 MIGLIOR COMBINAZIONE (per Accuracy raw):")
    print(f"   Min Samples = {best_acc['Min_Samples']}, Latent Dim = {best_acc['Latent_Dim']}")
    print(f"   Accuracy: {best_acc['Acc_Breed (mean)']:.4f}")
    print(f"   Num Razze: {best_acc.get('Num Classes Breed', 'N/A')}")

### 3.3 Analisi Trade-off: Numero Razze vs Performance

Al crescere di `min_samples`, diminuisce il numero di razze incluse ma aumenta la qualità dei dati per razza.

In [ ]:
# === ANALISI TRADE-OFF: NUMERO RAZZE vs PERFORMANCE ===
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Numero di razze per soglia min_samples
ax1 = axes[0, 0]
if 'Num Classes Breed' in df_grid.columns:
    breed_counts = df_grid.groupby('Min_Samples')['Num Classes Breed'].first()
    ax1.bar(breed_counts.index, breed_counts.values, color='steelblue', alpha=0.8, width=7)
    ax1.set_xlabel('Min Samples per Breed', fontsize=11)
    ax1.set_ylabel('Numero di Razze', fontsize=11)
    ax1.set_title('Razze incluse per soglia', fontweight='bold')
    ax1.set_xticks(MIN_SAMPLES_RANGE)
    for x, y in zip(breed_counts.index, breed_counts.values):
        ax1.text(x, y + 0.3, str(y), ha='center', fontweight='bold', fontsize=12)
    ax1.grid(True, alpha=0.3, axis='y')

# 2. Raw Accuracy media per min_samples
ax2 = axes[0, 1]
if 'Acc_Breed (mean)' in df_grid.columns:
    acc_by_samples = df_grid.groupby('Min_Samples')['Acc_Breed (mean)'].agg(['mean', 'std'])
    ax2.errorbar(acc_by_samples.index, acc_by_samples['mean'], 
                 yerr=acc_by_samples['std'], fmt='o-', capsize=5, 
                 color='coral', linewidth=2, markersize=8)
    ax2.set_xlabel('Min Samples per Breed', fontsize=11)
    ax2.set_ylabel('Accuracy Breed (media ± std)', fontsize=11)
    ax2.set_title('Raw Accuracy media per soglia\n(⚠️ non comparabile direttamente)', fontweight='bold')
    ax2.set_xticks(MIN_SAMPLES_RANGE)
    ax2.grid(True, alpha=0.3)

# 3. Cohen's Kappa per min_samples (confronto equo)
ax3 = axes[1, 0]
if 'Kappa_Breed (mean)' in df_grid.columns:
    kappa_by_samples = df_grid.groupby('Min_Samples')['Kappa_Breed (mean)'].agg(['mean', 'std'])
    ax3.errorbar(kappa_by_samples.index, kappa_by_samples['mean'], 
                 yerr=kappa_by_samples['std'], fmt='D-', capsize=5, 
                 color='purple', linewidth=2, markersize=8)
    ax3.set_xlabel('Min Samples per Breed', fontsize=11)
    ax3.set_ylabel("Cohen's Kappa (media ± std)", fontsize=11)
    ax3.set_title("Cohen's Kappa per soglia\n(✅ chance-adjusted, comparabile)", fontweight='bold')
    ax3.set_xticks(MIN_SAMPLES_RANGE)
    ax3.grid(True, alpha=0.3)

# 4. Kappa media per latent_dim
ax4 = axes[1, 1]
if 'Kappa_Breed (mean)' in df_grid.columns:
    kappa_by_dim = df_grid.groupby('Latent_Dim')['Kappa_Breed (mean)'].agg(['mean', 'std'])
    ax4.errorbar(kappa_by_dim.index, kappa_by_dim['mean'], 
                 yerr=kappa_by_dim['std'], fmt='s-', capsize=5, 
                 color='seagreen', linewidth=2, markersize=8)
    ax4.set_xlabel('Latent Dimension', fontsize=11)
    ax4.set_ylabel("Cohen's Kappa (media ± std)", fontsize=11)
    ax4.set_title("Cohen's Kappa per latent dim\n(media su tutti i min_samples)", fontweight='bold')
    ax4.set_xticks(LATENT_DIM_RANGE)
    ax4.grid(True, alpha=0.3)

plt.suptitle('📊 Trade-off Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tradeoff_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiche riassuntive
print("\n📊 STATISTICHE PER MIN_SAMPLES (media su tutte le latent dim):")
summary_cols = {'Num Classes Breed': 'first'}
if 'Acc_Breed (mean)' in df_grid.columns:
    summary_cols['Acc_Breed (mean)'] = ['mean', 'std', 'max']
if 'Kappa_Breed (mean)' in df_grid.columns:
    summary_cols['Kappa_Breed (mean)'] = ['mean', 'std', 'max']
summary = df_grid.groupby('Min_Samples').agg(summary_cols).round(4)
print(summary.to_string())

print("\n📊 STATISTICHE PER LATENT DIM (media su tutti i min_samples):")
dim_cols = {}
if 'Acc_Breed (mean)' in df_grid.columns:
    dim_cols['Acc_Breed (mean)'] = ['mean', 'std', 'max']
if 'Kappa_Breed (mean)' in df_grid.columns:
    dim_cols['Kappa_Breed (mean)'] = ['mean', 'std', 'max']
summary_dim = df_grid.groupby('Latent_Dim').agg(dim_cols).round(4)
print(summary_dim.to_string())

### 3.4 Linee di Tendenza per ogni Latent Dim

Analisi dettagliata dell'effetto del numero di sample per ogni dimensione latente.

In [ ]:
# === LINEE DI TENDENZA PER LATENT DIM ===
import matplotlib.pyplot as plt

metrics_lines = {
    'Kappa_Breed (mean)': "Cohen's Kappa (chance-adjusted)",
    'Acc_Breed (mean)': 'Raw Breed Accuracy',
    'Generalization (GE) (mean)': 'Generalization (GE)',
    'Silhouette (mean)': 'Silhouette Score',
}
metrics_lines = {k: v for k, v in metrics_lines.items() if k in df_grid.columns}

n_metrics = len(metrics_lines)
fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5))
if n_metrics == 1:
    axes = [axes]

colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

for idx, (metric_col, metric_name) in enumerate(metrics_lines.items()):
    ax = axes[idx]
    
    for i, latent_dim in enumerate(LATENT_DIM_RANGE):
        subset = df_grid[df_grid['Latent_Dim'] == latent_dim].sort_values('Min_Samples')
        ax.plot(subset['Min_Samples'], subset[metric_col], 
                'o-', label=f'LD={latent_dim}', color=colors[i],
                linewidth=2, markersize=8)
    
    ax.set_xlabel('Min Samples per Breed', fontsize=11)
    ax.set_ylabel(metric_name, fontsize=11)
    ax.set_title(metric_name, fontweight='bold')
    ax.set_xticks(MIN_SAMPLES_RANGE)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('📈 Trend per Dimensione Latente', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('trend_per_latent_dim.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === CONCLUSIONI GRIGLIA ESPERIMENTI ===
print("🎯 ANALISI DEI RISULTATI (20 combinazioni con 3-Fold CV)")
print("="*60)

if 'Kappa_Breed (mean)' in df_grid.columns:
    # Top 5 per Kappa (confronto equo)
    top5_kappa = df_grid.nlargest(5, 'Kappa_Breed (mean)')
    print("\n🏆 TOP 5 CONFIGURAZIONI (per Cohen's Kappa - confronto equo):")
    for i, (_, row) in enumerate(top5_kappa.iterrows(), 1):
        medal = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"][i-1]
        print(f"   {medal} Samples={int(row['Min_Samples'])}, LatDim={int(row['Latent_Dim'])} "
              f"→ Kappa={row['Kappa_Breed (mean)']:.4f}, Acc={row.get('Acc_Breed (mean)', 0):.4f} "
              f"({int(row.get('Num Classes Breed', 0))} razze)")

if 'Acc_Breed (mean)' in df_grid.columns:
    top5 = df_grid.nlargest(5, 'Acc_Breed (mean)')
    print("\n🏆 TOP 5 CONFIGURAZIONI (per Accuracy raw):")
    for i, (_, row) in enumerate(top5.iterrows(), 1):
        medal = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"][i-1]
        print(f"   {medal} Samples={int(row['Min_Samples'])}, LatDim={int(row['Latent_Dim'])} "
              f"→ Acc={row['Acc_Breed (mean)']:.4f} ({int(row.get('Num Classes Breed', 0))} razze)")

print("\n" + "="*60)
print("📝 NOTE:")
print("   • Tutti gli esperimenti usano SOLO sample reali (NO dati sintetici)")
print("   • Task: solo classificazione breed")
print("   • Le razze con più sample del minimo vengono cappate (downsampled)")
print("   • Cohen's Kappa è la metrica preferita per confronto equo tra soglie diverse")
print("   • 3-Fold Cross Validation per ogni esperimento (media ± std su 3 fold)")
print("   • Batch size fisso per tutti gli esperimenti")
print("   • Le confusion matrix sono in results/confusion_matrices/")

# Salva CSV con tutti i risultati (escludi colonne non serializzabili)
os.makedirs('results', exist_ok=True)
export_cols = [c for c in df_grid.columns if c not in ['Confusion Matrix Breed', 'Class Names Breed']]
df_grid[export_cols].to_csv('results/grid_experiment_results.csv', index=False)
print("\n💾 Risultati salvati in results/grid_experiment_results.csv")

### 3.5 Confusion Matrix per ogni Esperimento

Visualizzazione delle confusion matrix per tutti i 20 esperimenti, organizzate in griglia per min_samples.

In [ ]:
# === CONFUSION MATRIX PER OGNI ESPERIMENTO ===
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Organizza per min_samples: 1 figura per soglia, con 4 subplot (1 per latent_dim)
for min_samples in MIN_SAMPLES_RANGE:
    subset = [r for r in grid_results if r.get('Min_Samples') == min_samples]
    if not subset:
        continue
    
    n_dims = len(LATENT_DIM_RANGE)
    fig, axes = plt.subplots(1, n_dims, figsize=(7 * n_dims, 6))
    if n_dims == 1:
        axes = [axes]
    
    for idx, latent_dim in enumerate(LATENT_DIM_RANGE):
        ax = axes[idx]
        
        # Trova il risultato corrispondente
        res = next((r for r in subset if r.get('Latent_Dim') == latent_dim), None)
        
        if res is None or res.get('Confusion Matrix Breed') is None:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=20)
            ax.set_title(f'LatDim={latent_dim}', fontweight='bold')
            continue
        
        cm = res['Confusion Matrix Breed']
        class_names = res.get('Class Names Breed', [str(i) for i in range(len(cm))])
        
        # Normalizza per riga (percentuale per classe vera)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)  # gestisci divisione per 0
        
        # Heatmap
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                    xticklabels=class_names, yticklabels=class_names,
                    vmin=0, vmax=1, linewidths=0.3, linecolor='gray',
                    cbar_kws={'label': 'Recall per classe'})
        
        acc = res.get('Acc_Breed (mean)', 0)
        kappa = res.get('Kappa_Breed (mean)', 0)
        ax.set_title(f'LatDim={latent_dim}\nAcc={acc:.3f} | κ={kappa:.3f}', fontweight='bold', fontsize=11)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.tick_params(axis='both', labelsize=7)
    
    n_classes = subset[0].get('Num Classes Breed', '?')
    fig.suptitle(f'📊 Confusion Matrix — Min Samples = {min_samples} ({n_classes} razze)\n'
                 f'(Normalizzate per riga: recall per classe)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'results/confusion_matrices/cm_grid_samples_{min_samples}.png', dpi=150, bbox_inches='tight')
    plt.show()

print("✅ Confusion matrix visualizzate per tutte le 20 combinazioni")

## 4. Training Singolo Esperimento

Seleziona **un solo** valore di `MIN_SAMPLES` e `LATENT_DIM` per lanciare un singolo training con 3-Fold CV.
Utile per rieseguire una configurazione specifica o testare nuovi iperparametri senza rieseguire tutta la griglia.

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURA QUI
# ═══════════════════════════════════════════════════
SINGLE_MIN_SAMPLES = 30    # Numero minimo di campioni per razza
SINGLE_LATENT_DIM  = 96    # Dimensione del latent space
# ═══════════════════════════════════════════════════

print(f"Configurazione singolo esperimento:")
print(f"   Min Samples: {SINGLE_MIN_SAMPLES}")
print(f"   Latent Dim:  {SINGLE_LATENT_DIM}")

In [ ]:
# === TRAINING SINGOLO ESPERIMENTO ===

# 1. Carica DataModule
print(f"📦 Caricamento DataModule (min_samples={SINGLE_MIN_SAMPLES})...")
dm_single = GenomicDataModule(
    bed_path=config['bed_path'],
    batch_size=config['batch_size'],
    maf_thresh=config['maf_thresh'],
    geno_thresh=config['geno_thresh'],
    min_samples_per_class=SINGLE_MIN_SAMPLES,
    ld_pruning=config['ld_pruning'],
    ld_threshold=config['ld_threshold'],
    ld_window=config['ld_window'],
    val_split=config['val_split'],
    test_split=config['test_split'],
    metadata_path=config['metadata_path'],
    use_breed=True,
    use_continent=False,
    use_caseina=False,
    use_attitudine=False
)
dm_single.setup()
print(f"   ✅ {dm_single.num_classes_breed} razze, {dm_single.num_snps} SNPs\n")

# 2. Configura esperimento
single_config = config.copy()
single_config['latent_dim'] = SINGLE_LATENT_DIM
single_config['min_samples_per_class'] = SINGLE_MIN_SAMPLES
single_config['target_samples'] = SINGLE_MIN_SAMPLES

exp_name = f"Samples_{SINGLE_MIN_SAMPLES}_LatDim_{SINGLE_LATENT_DIM}"

# 3. Esegui training
print(f"🔬 Training: {exp_name}")
print(f"   K-Fold: {config['k_folds']} | Max Epochs: {config['max_epochs']}")
print(f"{'='*60}")

single_result = run_experiment(
    name=exp_name,
    config=single_config,
    balanced=False,
    cap_samples=True,
    coarse_mapping=None,
    max_epochs=config['max_epochs'],
    k_folds=config['k_folds'],
    classifier_config='breed_only',
    data_module=dm_single,
    save_checkpoint_path=f"checkpoints/{exp_name}.ckpt"
)

single_result['Min_Samples'] = SINGLE_MIN_SAMPLES
single_result['Latent_Dim'] = SINGLE_LATENT_DIM

print(f"\n✅ Training completato!")

In [ ]:
# === RISULTATI SINGOLO ESPERIMENTO ===
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print(f"📊 RISULTATI: {exp_name}")
print(f"{'='*60}")
print(f"   Razze:       {single_result.get('Num Classes Breed', '?')}")
print(f"   SNPs:        {single_result.get('Num SNPs', '?')}")
print(f"   Best Fold:   {single_result.get('Best Fold', '?')}")

# Stampa tutte le metriche (mean ± std)
print(f"\n   Metriche (mean ± std su {config['k_folds']} fold):")
for key in sorted(single_result.keys()):
    if key.endswith('(mean)'):
        std_key = key.replace('(mean)', '(std)')
        mean_val = single_result[key]
        std_val = single_result.get(std_key, 0)
        metric_name = key.replace(' (mean)', '')
        print(f"   {metric_name:30s}: {mean_val:.4f} ± {std_val:.4f}")

# Confusion Matrix
cm = single_result.get('Confusion Matrix Breed')
if cm is not None:
    class_names = single_result.get('Class Names Breed', [str(i) for i in range(len(cm))])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                vmin=0, vmax=1, linewidths=0.3, linecolor='gray',
                cbar_kws={'label': 'Recall per classe'})
    acc = single_result.get('Acc_Breed (mean)', 0)
    kappa = single_result.get('Kappa_Breed (mean)', 0)
    ax.set_title(f'{exp_name}\nAcc={acc:.3f} | κ={kappa:.3f}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='both', labelsize=7)
    plt.tight_layout()
    plt.show()

## 5. Feature Selection Validation — SHAP(VMGP)

Studio analogo al modello Contrastivo: verificare se i Top-% SNP identificati
dagli IG del classificatore breed del VMGP sono sufficienti per classificare
la razza meglio di una selezione casuale.

**Prerequisito:** eseguire prima la Sezione 4 (dm_single, SINGLE_MIN_SAMPLES, SINGLE_LATENT_DIM devono essere definiti).


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEZIONE 5 — SHAP SNP Pre-selection (protocollo paper VMGP)
# ═══════════════════════════════════════════════════════════════
import shap
import numpy as np
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from pytorch_lightning.callbacks import EarlyStopping
import pandas as pd
import os

from vae import VMGP_LightningSystem

# ── CONFIG (riusa il DataModule già caricato per min_samples=30) ──
SHAP_MIN_SAMPLES = 30
SHAP_LATENT_DIM  = 96       # usa la tua configurazione migliore
SHAP_EPOCHS      = 100
SEED             = 42
os.makedirs('results/shap_vmgp', exist_ok=True)

# ── 1. Carica DataModule dedicato per SHAP ────────────────────────
dm = GenomicDataModule(
    bed_path=BED_PATH,
    metadata_path=METADATA_PATH,
    batch_size=64,
    maf_thresh=0.01,
    geno_thresh=0.1,
    min_samples_per_class=SHAP_MIN_SAMPLES,
    ld_pruning=True,
    ld_threshold=0.2,
    ld_window=50,
    val_split=0.15,
    test_split=0.15,
    use_breed=True,
    use_continent=False,
    use_caseina=False,
    use_attitudine=False
)
dm.setup()

X_all = dm.X_processed    # (2976, 46453) float32
y_all = dm.y_breed_processed
print(f"Dataset completo: {X_all.shape[0]} campioni | {X_all.shape[1]} SNP | {dm.num_classes_breed} razze")

X_all = dm.X_processed                        # (2976, 46453) float32
y_all = dm.y_breed_processed                   # (2976,) int

print(f"Dataset completo: {X_all.shape[0]} campioni | {X_all.shape[1]} SNP | {dm.num_classes_breed} razze")

# ── 2. Split 10% ps / 90% validation (stratificato per razza) ────
ps_idx, val90_idx = train_test_split(
    np.arange(len(X_all)),
    test_size=0.90, stratify=y_all, random_state=SEED
)
X_ps,  y_ps  = X_all[ps_idx],     y_all[ps_idx]
X_90,  y_90  = X_all[val90_idx],  y_all[val90_idx]
print(f"ps (10%): {len(X_ps)} campioni | val90: {len(X_90)} campioni")

# ── 3. Wrapper deterministico per SHAP ───────────────────────────
class BreedWrapper(nn.Module):
    """SNP (float32) → logits_breed.
    In eval() mode il reparameterize restituisce mu, quindi è deterministico.
    SHAP calcola i gradienti attraverso: encoder → fc_mu → predictor_breed
    """
    def __init__(self, vmgp_network):
        super().__init__()
        self.net = vmgp_network

    def forward(self, x):
        encoded = self.net.encoder(x)
        mu      = self.net.fc_mu(encoded)
        return self.net.predictor_breed(mu)   # (B, num_classes_breed)

# ── 4. Training modello ps ────────────────────────────────────────
print("\n🔧 Training modello ps (10%)...")

# Dummy y_cont/cas/att (non usati — breed_only)
y_dummy = np.zeros(len(y_ps), dtype=np.int64)

ps_ds = TensorDataset(
    torch.FloatTensor(X_ps),
    torch.LongTensor(y_ps),
    torch.LongTensor(y_dummy),  # continent
    torch.LongTensor(y_dummy),  # caseina
    torch.LongTensor(y_dummy),  # attitudine
)
# Split interno 85/15 per l'early stopping
n_val_ps = max(int(0.15 * len(ps_ds)), 1)
n_train_ps = len(ps_ds) - n_val_ps
ps_train_ds, ps_val_ds = torch.utils.data.random_split(
    ps_ds, [n_train_ps, n_val_ps],
    generator=torch.Generator().manual_seed(SEED)
)

ps_train_loader = DataLoader(ps_train_ds, batch_size=32, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=True)
ps_val_loader   = DataLoader(ps_val_ds,   batch_size=32,
                              num_workers=4, pin_memory=True)

model_ps = VMGP_LightningSystem(
    num_snps=dm.num_snps,
    num_classes_breed=dm.num_classes_breed,
    num_classes_continent=0,
    num_classes_caseina=0,
    num_classes_attitudine=0,
    alpha=0.5, lr=0.0001,
    use_breed=True, use_continent=False, use_caseina=False, use_attitudine=False,
    weight_breed=1.0, weight_continent=0.0, weight_caseina=0.0, weight_attitudine=0.0,
    latent_dim=SHAP_LATENT_DIM
)

early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min', verbose=False)
trainer_ps = pl.Trainer(
    precision="16-mixed", max_epochs=SHAP_EPOCHS,
    accelerator='gpu', devices=1,
    callbacks=[early_stop],
    enable_progress_bar=True, enable_model_summary=False,
    enable_checkpointing=False, logger=False
)
trainer_ps.fit(model_ps, ps_train_loader, ps_val_loader)
print("✅ Training ps completato.")

# ── 5. Calcolo SHAP values ─────────────────────────────────────────
class BreedWrapper(nn.Module):
    """Usa il forward del LightningSystem direttamente — nessun accesso agli attributi interni."""
    def __init__(self, lightning_system):
        super().__init__()
        self.sys = lightning_system

    def forward(self, x):
        outputs = self.sys(x)          # dict con 'mu', 'logits_breed', 'x_recon', ...
        return outputs['logits_breed'] # (B, num_classes_breed)

wrapper = BreedWrapper(model_ps).eval().to('cuda')


X_ps_t = torch.FloatTensor(X_ps).to('cuda')
# Background: 100 campioni random dal ps (come da paper)
np.random.seed(SEED)
bg_idx  = np.random.choice(len(X_ps), min(100, len(X_ps)), replace=False)
X_bg_t  = X_ps_t[bg_idx]

print(f"\n🔍 Calcolo SHAP su {len(X_ps)} campioni ps...")
explainer   = shap.GradientExplainer(wrapper, X_bg_t)
shap_values = explainer.shap_values(X_ps_t)   # list o ndarray, dipende dalla versione SHAP

# ── 6. Aggregazione robusta: output finale sempre (n_snps,) ───────
def aggregate_shap_to_global(shap_values_obj, n_features: int) -> np.ndarray:
    if isinstance(shap_values_obj, list):
        arr = np.stack([np.asarray(v) for v in shap_values_obj], axis=0)
    else:
        arr = np.asarray(shap_values_obj)

    feature_axes = [i for i, size in enumerate(arr.shape) if size == n_features]
    if not feature_axes:
        raise ValueError(
            f"Impossibile identificare asse feature in shape {arr.shape} (n_features={n_features})"
        )

    arr = np.moveaxis(arr, feature_axes[0], -1)
    shap_global_scores = np.abs(arr).mean(axis=tuple(range(arr.ndim - 1)))
    return np.asarray(shap_global_scores, dtype=np.float64)

N_SNPS = X_all.shape[1]
shap_global = aggregate_shap_to_global(shap_values, N_SNPS)

np.save('results/shap_vmgp/shap_global.npy', shap_global)
print(f"✅ SHAP global salvato. Shape: {shap_global.shape}")

# Top 10 SNP
top10 = np.argsort(shap_global)[::-1][:10]
print("\nTop 10 SNP per SHAP aggregato:")
for r, idx in enumerate(top10, 1):
    print(f"  {r:2d}. SNP[{int(idx)}] = {float(shap_global[idx]):.6f}")

# ── 7. Curva accuracy per soglia di SNP ──────────────────────────
K_SNPS = [47, 233, 465, 930, 2323, 4646]   # 0.1%, 0.5%, 1%, 2%, 5%, 10%

classifiers = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'LR':  LogisticRegression(max_iter=1000, solver='lbfgs',
                               multi_class='multinomial', random_state=SEED),
    'RF':  RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
}
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

# Baseline full sul 90%
print("\n📊 Baseline full (90% dataset)...")
X90_s = StandardScaler().fit_transform(X_90)
for name, clf in classifiers.items():
    s = cross_val_score(clf, X90_s, y_90, cv=skf, scoring='accuracy', n_jobs=-1).mean()
    print(f"  {name}: {s:.4f}")

results = []
for k in K_SNPS:
    top_idx = np.argsort(shap_global)[::-1][:k]
    X_sub   = StandardScaler().fit_transform(X_90[:, top_idx])
    row = {'k': k, 'pct': f"{k/N_SNPS*100:.1f}%"}

    # Random baseline (5 seed)
    rand_acc = {n: [] for n in classifiers}
    for seed in range(5):
        ri = np.random.RandomState(seed).choice(N_SNPS, k, replace=False)
        Xr = StandardScaler().fit_transform(X_90[:, ri])
        for name, clf in classifiers.items():
            rand_acc[name].append(
                cross_val_score(clf, Xr, y_90, cv=skf, scoring='accuracy', n_jobs=-1).mean()
            )

    for name, clf in classifiers.items():
        s = cross_val_score(clf, X_sub, y_90, cv=skf, scoring='accuracy', n_jobs=-1).mean()
        row[f'acc_{name}_shap'] = s
        row[f'acc_{name}_rand'] = np.mean(rand_acc[name])
        print(f"Top {k:5d} ({row['pct']:>5s}): {name} SHAP={s:.4f} | rand={np.mean(rand_acc[name]):.4f}")
    results.append(row)

df_shap = pd.DataFrame(results)
df_shap.to_csv('results/shap_vmgp/accuracy_curve.csv', index=False)
print("\n💾 Salvato: results/shap_vmgp/accuracy_curve.csv")



CARICAMENTO DATASET PLINK: ../data/ADAPTmap_genotypeTOP_20160222_full.bed
📋 Metadata caricato: 105 razze
📊 Shape iniziale: (4653, 53347) (4653 campioni, 53347 SNP)

🏷️  Estrazione label multiple...
   Continent: 3960/4653 campioni con label
   Caseina: 1191/4653 campioni con label

🔍 Filtro Razze Rare (min 30 campioni):
   Razze valide: 34/144
   Campioni rimossi: 1677

🧬 QUALITY CONTROL PIPELINE
────────────────────────────────────────────────────────────
1️⃣  Filtro Missingness SNP...
    ✓ Rimossi 2142 SNP con missingness >10.0%
2️⃣  Imputazione valori mancanti (media)...
    ✓ Imputazione completata
3️⃣  Filtro MAF (Minor Allele Frequency)...
    ✓ Rimossi 4 SNP con MAF <0.01
4️⃣  LD Pruning (Window=50, r²>0.2)...
    Device: cuda
    ...10000/51201 SNP scansionati
    ...20000/51201 SNP scansionati
    ...30000/51201 SNP scansionati
    ...40000/51201 SNP scansionati
    ✓ Rimossi 4748 SNP ridondanti (LD)

📈 DIMENSIONI FINALI: (2976, 46453)

🏷️  Classi per task:
   Breed: 34
   C

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 99: 100%|██████████| 7/7 [00:00<00:00, 15.89it/s, train_acc_breed=1.000, train_loss=0.233, val_acc_breed=0.864, val_loss=0.670]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 7/7 [00:00<00:00, 15.87it/s, train_acc_breed=1.000, train_loss=0.233, val_acc_breed=0.864, val_loss=0.670]
✅ Training ps completato.

🔍 Calcolo SHAP su 297 campioni ps...


# Random forest approach

In [ ]:
# ── 2. Addestra RF su 10% ps E su 80% training ───────────────────
import warnings

# Filtra solo il warning di deprecazione sklearn su multi_class
warnings.filterwarnings(
    'ignore',
    message=".*'multi_class' was deprecated.*",
    category=FutureWarning
 )

# Assicura la presenza della directory di output
os.makedirs('results/rf_importance', exist_ok=True)

# Split 80/20 standard (indipendente dallo split 10/90 del SHAP)
train80_idx, test20_idx = train_test_split(
    np.arange(len(X_all)),
    test_size=0.20, stratify=y_all, random_state=SEED
)
X_train80, y_train80 = X_all[train80_idx], y_all[train80_idx]
X_test20,  y_test20  = X_all[test20_idx],  y_all[test20_idx]

# RF sul 10% ps
print(f"Training RF su {len(X_ps)} campioni (10% ps)...")
rf_10 = RandomForestClassifier(n_estimators=500, max_features='sqrt',
                                random_state=SEED, n_jobs=-1)
rf_10.fit(X_ps, y_ps)
rf_importance_10 = rf_10.feature_importances_
np.save('results/rf_importance/rf_importance_10pct.npy', rf_importance_10)
print(f"✅ RF 10%: SNP con imp>0: {(rf_importance_10 > 0).sum()}")

# RF sull'80%
print(f"Training RF su {len(X_train80)} campioni (80% train)...")
rf_80 = RandomForestClassifier(n_estimators=500, max_features='sqrt',
                                random_state=SEED, n_jobs=-1)
rf_80.fit(X_train80, y_train80)
rf_importance_80 = rf_80.feature_importances_
np.save('results/rf_importance/rf_importance_80pct.npy', rf_importance_80)
print(f"✅ RF 80%: SNP con imp>0: {(rf_importance_80 > 0).sum()}")

# ── 3. Curva accuracy per entrambi ───────────────────────────────
# Per RF 10%: valuta sul 90% (stesso split SHAP → confronto diretto)
# Per RF 80%: valuta sul 20% restante (split indipendente)

# LR senza parametro multi_class (deprecato in sklearn >= 1.5)
classifiers_clean = dict(classifiers)
classifiers_clean['LR'] = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    random_state=SEED
)

configs = [
    ('RF_10pct', rf_importance_10, X_90,     y_90),
    ('RF_80pct', rf_importance_80, X_test20, y_test20),
]

results = []
for label, importance, X_eval, y_eval in configs:
    print(f"\n📊 Curva accuracy — {label} (eval su {len(X_eval)} campioni):")
    for k in K_SNPS:
        top_idx = np.argsort(importance)[::-1][:k]
        X_sub   = StandardScaler().fit_transform(X_eval[:, top_idx])
        row = {'method': label, 'k': k, 'pct': f"{k/N_SNPS*100:.1f}%"}

        for name, clf in classifiers_clean.items():
            s = cross_val_score(clf, X_sub, y_eval, cv=skf,
                                scoring='accuracy', n_jobs=-1).mean()
            row[f'acc_{name}'] = s
            print(f"  Top {k:5d} ({row['pct']:>5s}): {name}={s:.4f}")
        results.append(row)

df_rf = pd.DataFrame(results)
df_rf.to_csv('results/rf_importance/accuracy_curve_rf.csv', index=False)
print("\n💾 Salvato: results/rf_importance/accuracy_curve_rf.csv")
